# EQ-VAE 潜空间交互式 Playground

这个 notebook 是自包含的可视化入口，可以直接在 Jupyter 里运行。

可以完成这些操作：

- 从 `/data/shared` 的 CIFAR-10 中按随机种子随机选图，或用索引指定多张图。
- 使用多个 Hugging Face VAE，例如 `zelaki/eq-vae`、`zelaki/eq-vae-ema`、`stabilityai/sd-vae-ft-mse`，并通过 `E_eqvae/D_eqvae/E_sdvae/D_sdvae` 直接操作。
- 使用本工程的 LDM `.ckpt` VAE，例如你训练出的 EQ-VAE checkpoint。
- 操作输入图像：旋转、翻转、缩放。
- 操作 latent：整体缩放、加噪、通道偏移、空间平移。
- 可视化 reconstruction、latent PCA、latent 前 3 通道、两图 latent 插值。

建议从上到下运行。第一次使用 Hugging Face 模型时可能需要联网下载权重。


In [ ]:
from pathlib import Path
import re
import sys
import math
from dataclasses import dataclass
from types import SimpleNamespace
from typing import List, Tuple

from IPython.display import display
from PIL import Image, ImageDraw
import numpy as np
import torch
import torch.nn.functional as F

# 让 notebook 无论从仓库根目录还是 notebooks/ 目录启动，都能找到项目代码。
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "train_eqvae").exists() else CWD.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "train_eqvae"))

from baselines.visual_adapters import RAE_SPECS as RAE_BASELINE_SPECS, get_rae_status, load_rae_adapter

DTYPES = {
    "fp16": torch.float16,
    "bf16": torch.bfloat16,
    "fp32": torch.float32,
}

# 这个 HF 权重在当前环境下用 fp16 会产生 NaN，显示时会退化成黑图。
HF_FORCE_FP32_MODELS = {"zelaki/eq-vae-ema"}


@dataclass
class VAEAdapter:
    source: str
    model: torch.nn.Module
    scaling_factor: float
    device: torch.device
    dtype: torch.dtype

    @torch.no_grad()
    def encode(self, x: torch.Tensor, posterior: str = "sample") -> torch.Tensor:
        self.model.eval()
        x = x.to(device=self.device, dtype=self.dtype)
        if self.source == "hf":
            latent_dist = self.model.encode(x).latent_dist
            z = latent_dist.mode() if posterior == "mode" else latent_dist.sample()
        else:
            dist = self.model.encode(x)
            z = dist.mode() if posterior == "mode" else dist.sample()
        return z * self.scaling_factor

    @torch.no_grad()
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        self.model.eval()
        z = z.to(device=self.device, dtype=self.dtype) / self.scaling_factor
        if self.source == "hf":
            x = self.model.decode(z).sample
        else:
            x = self.model.decode(z)
        return x.clamp(-1, 1)


def resolve_device(device_arg: str) -> torch.device:
    if device_arg == "cuda" and not torch.cuda.is_available():
        print("未检测到 CUDA，自动改用 CPU。")
        return torch.device("cpu")
    return torch.device(device_arg)


def resolve_dtype(dtype_arg: str, device: torch.device) -> torch.dtype:
    dtype = DTYPES[dtype_arg]
    if device.type == "cpu" and dtype != torch.float32:
        print("CPU 上自动使用 fp32，避免半精度算子不兼容。")
        return torch.float32
    return dtype


def default_hf_scaling_factor(model_name: str, config_scaling) -> float:
    if config_scaling is not None:
        return float(config_scaling)
    if model_name in {"stabilityai/sd-vae-ft-mse", "stabilityai/sd-vae-ft-ema"}:
        return 0.18215
    return 1.0


def load_hf_vae(args, device: torch.device, dtype: torch.dtype) -> VAEAdapter:
    from diffusers.models import AutoencoderKL

    model_dtype = torch.float32 if args.hf_model_name in HF_FORCE_FP32_MODELS else dtype
    if model_dtype != dtype:
        print(f"{args.hf_model_name} 在半精度下会产生 NaN，已自动改用 fp32。")

    vae = AutoencoderKL.from_pretrained(args.hf_model_name)
    scaling = args.vae_scaling_factor
    if scaling is None:
        scaling = default_hf_scaling_factor(args.hf_model_name, getattr(vae.config, "scaling_factor", None))
    vae = vae.to(device=device, dtype=model_dtype).eval()
    return VAEAdapter("hf", vae, float(scaling), device, model_dtype)


def load_ldm_vae(args, device: torch.device, dtype: torch.dtype) -> VAEAdapter:
    if args.ldm_ckpt is None:
        raise ValueError("vae_source='ldm' 时必须提供 ldm_ckpt。")
    if args.ldm_config is None:
        raise ValueError("vae_source='ldm' 时必须提供 ldm_config。")

    from omegaconf import OmegaConf
    from train_eqvae.ldm.models.autoencoder import AutoencoderKL as LDMAutoencoderKL

    config = OmegaConf.load(args.ldm_config)
    vae = LDMAutoencoderKL(**config.model.params)
    ckpt = torch.load(args.ldm_ckpt, map_location="cpu")
    state_dict = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    missing, unexpected = vae.load_state_dict(state_dict, strict=False)
    print(f"LDM VAE 已加载：missing={len(missing)} unexpected={len(unexpected)}")

    scaling = args.vae_scaling_factor if args.vae_scaling_factor is not None else 0.18215
    vae = vae.to(device=device, dtype=dtype).eval()
    return VAEAdapter("ldm", vae, float(scaling), device, dtype)


def load_vae(args, device: torch.device, dtype: torch.dtype) -> VAEAdapter:
    if args.vae_source == "hf":
        return load_hf_vae(args, device, dtype)
    return load_ldm_vae(args, device, dtype)


def center_crop_resize(img: Image.Image, size: int) -> Image.Image:
    w, h = img.size
    scale = size / min(w, h)
    resized = img.resize((round(w * scale), round(h * scale)), Image.Resampling.BICUBIC)
    left = (resized.width - size) // 2
    top = (resized.height - size) // 2
    return resized.crop((left, top, left + size, top + size))


def apply_canvas_scale(img: Image.Image, scale: float) -> Image.Image:
    if abs(scale - 1.0) < 1e-6:
        return img

    size = img.size[0]
    scaled_size = max(1, round(size * scale))
    scaled = img.resize((scaled_size, scaled_size), Image.Resampling.BICUBIC)
    if scale >= 1.0:
        left = (scaled.width - size) // 2
        top = (scaled.height - size) // 2
        return scaled.crop((left, top, left + size, top + size))

    canvas = Image.new("RGB", (size, size), (127, 127, 127))
    left = (size - scaled.width) // 2
    top = (size - scaled.height) // 2
    canvas.paste(scaled, (left, top))
    return canvas


def apply_input_transforms(img: Image.Image, args) -> Image.Image:
    img = img.convert("RGB")
    img = center_crop_resize(img, args.image_size)

    if args.input_flip == "h":
        img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    elif args.input_flip == "v":
        img = img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

    if args.input_rotate:
        img = img.rotate(args.input_rotate, resample=Image.Resampling.BICUBIC)

    img = apply_canvas_scale(img, args.input_scale)
    return img


def prepare_image(path: str, args) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return apply_input_transforms(img, args)


def image_to_tensor(img: Image.Image, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    arr = np.asarray(img).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = x.to(device=device, dtype=dtype)
    return x * 2.0 - 1.0


def tensor_to_pil_m11(x: torch.Tensor) -> Image.Image:
    x = x[0].detach().float().cpu().clamp(-1, 1)
    x = ((x + 1.0) * 127.5).round().byte()
    return Image.fromarray(x.permute(1, 2, 0).numpy(), mode="RGB")


def tensor_to_pil_01(x: torch.Tensor) -> Image.Image:
    x = x[0].detach().float().cpu().clamp(0, 1)
    x = (x * 255.0).round().byte()
    return Image.fromarray(x.permute(1, 2, 0).numpy(), mode="RGB")


def pca_to_rgb(latents: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    b, c, h, w = latents.shape
    if c == 0:
        return torch.zeros((b, 3, h, w), dtype=torch.float32)

    z = torch.nan_to_num(latents.detach().float().cpu())
    x = z.permute(0, 2, 3, 1).reshape(-1, c)
    x = x - x.mean(dim=0, keepdim=True)

    if x.numel() == 0 or x.abs().max().item() < eps:
        return first3_to_rgb(z, eps=eps).cpu()

    try:
        _, _, vh = torch.linalg.svd(x, full_matrices=False)
        top = vh[:min(3, c)].T
        projected = x @ top
    except RuntimeError:
        return first3_to_rgb(z, eps=eps).cpu()

    if projected.shape[1] < 3:
        pad = torch.zeros(projected.shape[0], 3 - projected.shape[1], dtype=projected.dtype)
        projected = torch.cat([projected, pad], dim=1)

    rgb = projected.reshape(b, h, w, 3).permute(0, 3, 1, 2)
    rgb = torch.nan_to_num(rgb)
    rgb_min = rgb.flatten(1).amin(dim=1).reshape(b, 1, 1, 1)
    rgb_max = rgb.flatten(1).amax(dim=1).reshape(b, 1, 1, 1)
    return (rgb - rgb_min) / (rgb_max - rgb_min).clamp_min(eps)


def first3_to_rgb(latents: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    z = latents.detach().float()
    if z.shape[1] < 3:
        pad = torch.zeros(z.shape[0], 3 - z.shape[1], z.shape[2], z.shape[3], device=z.device)
        z = torch.cat([z, pad], dim=1)
    rgb = z[:, :3]
    b = rgb.shape[0]
    rgb_min = rgb.reshape(b, -1).min(dim=1)[0].reshape(b, 1, 1, 1)
    rgb_max = rgb.reshape(b, -1).max(dim=1)[0].reshape(b, 1, 1, 1)
    return (rgb - rgb_min) / (rgb_max - rgb_min + eps)


def parse_float_list(value: str) -> List[float]:
    if not value:
        return []
    return [float(x.strip()) for x in value.split(",") if x.strip()]


def parse_translation_list(value: str) -> List[Tuple[int, int]]:
    if not value:
        return []
    translations = []
    for item in value.split(";"):
        item = item.strip()
        if not item:
            continue
        dx, dy = [int(v.strip()) for v in item.split(",", 1)]
        translations.append((dx, dy))
    return translations


def rotate_latent(z: torch.Tensor, degrees: float) -> torch.Tensor:
    normalized = degrees % 360
    quarter_turn = normalized / 90.0
    nearest = round(quarter_turn)
    if abs(quarter_turn - nearest) < 1e-6:
        return torch.rot90(z, k=nearest % 4, dims=(-2, -1))

    angle = math.radians(float(degrees))
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    theta = torch.tensor(
        [[cos_a, -sin_a, 0.0], [sin_a, cos_a, 0.0]],
        device=z.device,
        dtype=torch.float32,
    ).unsqueeze(0).repeat(z.shape[0], 1, 1)
    z_float = z.float()
    grid = F.affine_grid(theta, z_float.shape, align_corners=False)
    return F.grid_sample(z_float, grid, mode="bilinear", padding_mode="zeros", align_corners=False).to(dtype=z.dtype)


def translate_latent(z: torch.Tensor, dx: int, dy: int) -> torch.Tensor:
    out = torch.zeros_like(z)
    height, width = z.shape[-2:]

    src_x0 = max(0, -dx)
    src_x1 = min(width, width - dx)
    dst_x0 = max(0, dx)
    dst_x1 = min(width, width + dx)

    src_y0 = max(0, -dy)
    src_y1 = min(height, height - dy)
    dst_y0 = max(0, dy)
    dst_y1 = min(height, height + dy)

    if src_x0 >= src_x1 or src_y0 >= src_y1:
        return out

    out[..., dst_y0:dst_y1, dst_x0:dst_x1] = z[..., src_y0:src_y1, src_x0:src_x1]
    return out


def parse_channel_shifts(value: str) -> List[Tuple[int, float]]:
    if not value:
        return []
    shifts = []
    for item in value.split(","):
        if not item.strip():
            continue
        channel, amount = item.split(":")
        shifts.append((int(channel), float(amount)))
    return shifts


def parse_indices(value: str) -> List[int]:
    if not value:
        return []
    indices = []
    for item in value.split(","):
        item = item.strip()
        if not item:
            continue
        if ":" in item:
            start, stop = item.split(":", 1)
            indices.extend(range(int(start), int(stop)))
        else:
            indices.append(int(item))
    return indices


def sanitize_name(name: str) -> str:
    name = name.replace(" ", "_")
    return re.sub(r"[^0-9A-Za-z_.+-]+", "_", name)


def load_cifar_dataset(name: str, root: str, split: str, download: bool = False):
    from torchvision.datasets import CIFAR10, CIFAR100

    train = split == "train"
    if name == "cifar10":
        return CIFAR10(root=root, train=train, download=download)
    if name == "cifar100":
        return CIFAR100(root=root, train=train, download=download)
    raise ValueError(f"不支持的数据集：{name}")


def select_indices(total: int, num_images: int, seed: int, explicit_indices: str = "") -> List[int]:
    indices = parse_indices(explicit_indices)
    if indices:
        for idx in indices:
            if idx < 0 or idx >= total:
                raise IndexError(f"索引 {idx} 越界，数据集大小为 {total}")
        return indices

    count = min(max(1, num_images), total)
    rng = np.random.default_rng(seed if seed is not None else None)
    return [int(i) for i in rng.choice(total, size=count, replace=False)]


def load_input_images(args) -> List[Tuple[str, Image.Image]]:
    if args.input_source == "file":
        if not args.image:
            raise ValueError("input_source='file' 时必须提供 image_path。")
        return [(Path(args.image).stem, prepare_image(args.image, args))]

    dataset = load_cifar_dataset(
        args.cifar_name,
        args.dataset_root,
        args.cifar_split,
        download=args.download_cifar,
    )
    indices = select_indices(len(dataset), args.num_images, args.seed, args.indices)
    samples = []
    for idx in indices:
        img, label = dataset[idx]
        name = f"{args.cifar_name}_{args.cifar_split}_idx{idx}_label{label}"
        samples.append((name, apply_input_transforms(img, args)))
    return samples


def save_variant(output_dir, name: str, img: Image.Image, variants: List[Tuple[str, Image.Image]]) -> None:
    if output_dir is not None:
        filename = sanitize_name(name) + ".png"
        img.save(output_dir / filename)
    variants.append((name, img))


def make_grid(variants: List[Tuple[str, Image.Image]], columns: int, cell_size: int, label_h: int = 28) -> Image.Image:
    if not variants:
        raise ValueError("没有可视化结果可生成网格。")

    columns = max(1, columns)
    rows = (len(variants) + columns - 1) // columns
    grid = Image.new("RGB", (columns * cell_size, rows * (cell_size + label_h)), (245, 245, 245))
    draw = ImageDraw.Draw(grid)

    for idx, (name, img) in enumerate(variants):
        row = idx // columns
        col = idx % columns
        x = col * cell_size
        y = row * (cell_size + label_h)
        draw.rectangle((x, y, x + cell_size, y + label_h), fill=(32, 32, 32))
        draw.text((x + 6, y + 7), name[:42], fill=(255, 255, 255))
        thumb = img.resize((cell_size, cell_size), Image.Resampling.BILINEAR)
        grid.paste(thumb, (x, y + label_h))
    return grid


def add_latent_variants(z: torch.Tensor, vae: VAEAdapter, args, output_dir: Path, variants: List[Tuple[str, Image.Image]]) -> None:
    for scale in parse_float_list(args.latent_scales):
        decoded = vae.decode(z * scale)
        save_variant(output_dir, f"latent_scale_{scale:g}", tensor_to_pil_m11(decoded), variants)

    if args.seed is not None:
        torch.manual_seed(args.seed)
    for strength in parse_float_list(args.latent_noises):
        noisy = z + torch.randn_like(z) * strength
        decoded = vae.decode(noisy)
        save_variant(output_dir, f"latent_noise_{strength:g}", tensor_to_pil_m11(decoded), variants)

    for channel, amount in parse_channel_shifts(args.channel_shifts):
        if channel < 0 or channel >= z.shape[1]:
            print(f"跳过 channel {channel}：latent 只有 {z.shape[1]} 个通道。")
            continue
        shifted = z.clone()
        shifted[:, channel] += amount
        decoded = vae.decode(shifted)
        save_variant(output_dir, f"channel_{channel}_shift_{amount:g}", tensor_to_pil_m11(decoded), variants)

    for degrees in parse_float_list(args.latent_rotations):
        rotated = rotate_latent(z, degrees)
        decoded = vae.decode(rotated)
        save_variant(output_dir, f"latent_rotate_{degrees:g}deg", tensor_to_pil_m11(decoded), variants)

    for dx, dy in parse_translation_list(args.latent_translations):
        translated = translate_latent(z, dx, dy)
        decoded = vae.decode(translated)
        save_variant(output_dir, f"latent_translate_x{dx}_y{dy}", tensor_to_pil_m11(decoded), variants)

    if args.latent_roll:
        dx, dy = [int(v.strip()) for v in args.latent_roll.split(",")]
        rolled = torch.roll(z, shifts=(dy, dx), dims=(-2, -1))
        decoded = vae.decode(rolled)
        save_variant(output_dir, f"latent_roll_x{dx}_y{dy}", tensor_to_pil_m11(decoded), variants)


def add_interpolation_variants(z_a: torch.Tensor, vae: VAEAdapter, args, output_dir: Path, variants: List[Tuple[str, Image.Image]]) -> None:
    if args.image_b is None:
        return
    image_b = prepare_image(args.image_b, args)
    x_b = image_to_tensor(image_b, vae.device, vae.dtype)
    z_b = vae.encode(x_b, posterior=args.posterior)
    steps = max(2, args.interp_steps)
    for i, alpha in enumerate(torch.linspace(0, 1, steps, device=z_a.device)):
        mixed = z_a * (1 - alpha) + z_b * alpha
        decoded = vae.decode(mixed)
        save_variant(output_dir, f"interp_{i:02d}_a{float(alpha):.2f}", tensor_to_pil_m11(decoded), variants)


def process_one_image(name: str, image: Image.Image, vae: VAEAdapter, args, output_dir) -> Image.Image:
    multi_sample = args.input_source == "cifar" and (args.num_images > 1 or bool(args.indices))
    sample_dir = None
    if output_dir is not None:
        sample_dir = output_dir / sanitize_name(name) if multi_sample else output_dir
        sample_dir.mkdir(parents=True, exist_ok=True)

    x = image_to_tensor(image, vae.device, vae.dtype)
    z = vae.encode(x, posterior=args.posterior)
    reconstruction = vae.decode(z)

    variants: List[Tuple[str, Image.Image]] = []
    save_variant(sample_dir, "00_processed_input", image, variants)
    save_variant(sample_dir, "01_reconstruction", tensor_to_pil_m11(reconstruction), variants)
    save_variant(sample_dir, "02_latent_pca", tensor_to_pil_01(pca_to_rgb(z)), variants)
    save_variant(sample_dir, "03_latent_first3", tensor_to_pil_01(first3_to_rgb(z)), variants)

    add_latent_variants(z, vae, args, sample_dir, variants)
    add_interpolation_variants(z, vae, args, sample_dir, variants)

    if args.save_latents and sample_dir is not None:
        torch.save(
            {
                "latent": z.detach().cpu(),
                "name": name,
                "input_source": args.input_source,
                "vae_source": args.vae_source,
                "hf_model_name": args.hf_model_name,
                "ldm_config": args.ldm_config,
                "ldm_ckpt": args.ldm_ckpt,
                "scaling_factor": vae.scaling_factor,
                "posterior": args.posterior,
            },
            sample_dir / "base_latent.pt",
        )

    grid = make_grid(variants, columns=args.grid_columns, cell_size=args.grid_cell_size)
    print(f"[{name}] 输入张量：{tuple(x.shape)}，latent：{tuple(z.shape)}")
    if sample_dir is not None:
        grid.save(sample_dir / "grid.png")
        print(f"[{name}] 总览图：{sample_dir / 'grid.png'}")
    else:
        print(f"[{name}] 总览图已生成但未保存。")
    return grid



print(f"项目根目录: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")


## 1. 配置参数

先改这里。路径可以写绝对路径，也可以写相对仓库根目录的路径。默认从 `/data/shared` 的 CIFAR-10 训练集中按 `seed` 随机选择 4 张。


In [ ]:
# 输入来源："file" 或 "cifar"
input_source = "cifar"

# 文件输入：input_source="file" 时使用。
image_path = "/path/to/image.jpg"

# CIFAR 输入：input_source="cifar" 时使用。
dataset_root = "/data/shared"
cifar_name = "cifar10"      # "cifar10" 或 "cifar100"
cifar_split = "train"       # "train" 或 "test"
num_images = 4               # 随机选择几张
indices = ""                 # 显式选择索引，例如 "0,5,9" 或 "10:20"；留空则按 seed 随机
# 如果 /data/shared 已经有 CIFAR-10，保持 False；缺数据时可以改成 True。
download_cifar = False

# 可选：第二张图像，用于 latent 插值；不需要就留空字符串。
image_b_path = ""

# 默认不写入 outputs 文件夹；需要保存图片时再改成 True。
save_outputs = False
# save_outputs=True 时才会用到这个目录。
output_dir = "outputs/notebook_latent_playground"

# VAE 来源："hf" 或 "ldm"
vae_source = "hf"

# Hugging Face VAE 示例：
# - "zelaki/eq-vae"
# - "zelaki/eq-vae-ema"
# - "stabilityai/sd-vae-ft-mse"
hf_model_name = "zelaki/eq-vae"

# LDM checkpoint VAE 设置。vae_source="ldm" 时需要 ldm_ckpt。
ldm_config = "train_eqvae/configs/eqvae_config.yaml"
ldm_ckpt = ""

# RAE 查看后端。默认不自动 clone/download；需要时再打开，避免一运行就下载大模型。
rae_repo_path = "external/RAE"
rae_auto_clone = False
rae_auto_download = False
rae_dtype_name = "fp32"

# 项目训练时的 exact SD-VAE baseline；运行 train_eqvae/download_sdvae.sh 后把路径指到 kl-f8/model.ckpt。
sdvae_klf8_ckpt = "pretrained_models/kl-f8/model.ckpt"

# None 表示：HF 从 config 读取，LDM 默认 0.18215。
vae_scaling_factor = None

# posterior="sample" 会有随机性；posterior="mode" 更稳定。
posterior = "sample"

# 推理设置
device_name = "cuda:3"     # "cuda", "cuda:0", "cpu"
dtype_name = "fp32"      # 固定用 fp32，最稳；代价是更慢、更占显存
seed = 0

# 输入图像操作
image_size = 256
input_rotate = 0.0       # 角度，例如 30
input_flip = "none"      # "none", "h", "v"
input_scale = 1.0        # >1 放大，<1 缩小并灰底填充

# latent 操作
latent_scales = "0.5,1.0,1.5"
latent_noises = "0.05,0.1"
channel_shifts = ""                    # 例如 "0:1.0,1:-1.0"
latent_rotations = "90,180,270"        # latent 层旋转角度；90 的倍数会用精确 rot90
latent_translations = "4,0;0,4;-4,0;0,-4"  # 零填充平移；格式 dx,dy;dx,dy，单位是 latent 像素
latent_roll = ""                       # 循环平移；例如 "2,0"，会从另一边卷回来
interp_steps = 5

# 总览图设置
grid_columns = 4
grid_cell_size = 256
save_latents = False


In [ ]:
def resolve_path(value):
    if value is None or value == "":
        return None
    p = Path(value)
    return str(p if p.is_absolute() else (ROOT / p).resolve())

args = SimpleNamespace(
    input_source=input_source,
    image=resolve_path(image_path),
    image_b=resolve_path(image_b_path),
    output_dir=resolve_path(output_dir),
    save_outputs=save_outputs,
    image_size=image_size,
    device=device_name,
    dtype=dtype_name,
    seed=seed,
    dataset_root=dataset_root,
    cifar_name=cifar_name,
    cifar_split=cifar_split,
    num_images=num_images,
    indices=indices,
    download_cifar=download_cifar,
    vae_source=vae_source,
    hf_model_name=hf_model_name,
    ldm_config=resolve_path(ldm_config),
    ldm_ckpt=resolve_path(ldm_ckpt),
    rae_repo_path=resolve_path(rae_repo_path),
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    rae_dtype=rae_dtype_name,
    sdvae_klf8_ckpt=resolve_path(sdvae_klf8_ckpt),
    vae_scaling_factor=vae_scaling_factor,
    posterior=posterior,
    input_rotate=input_rotate,
    input_flip=input_flip,
    input_scale=input_scale,
    latent_scales=latent_scales,
    latent_noises=latent_noises,
    channel_shifts=channel_shifts,
    latent_rotations=latent_rotations,
    latent_translations=latent_translations,
    latent_roll=latent_roll,
    interp_steps=interp_steps,
    save_latents=save_latents,
    grid_columns=grid_columns,
    grid_cell_size=grid_cell_size,
)

if args.input_source == "file" and (args.image is None or not Path(args.image).exists()):
    raise FileNotFoundError(f"请先把 image_path 改成真实图片路径。当前值: {image_path}")

if args.seed is not None and args.seed >= 0:
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
else:
    args.seed = None

device = resolve_device(args.device)
dtype = resolve_dtype(args.dtype, device)
rae_dtype = resolve_dtype(args.rae_dtype, device)
out_dir = Path(args.output_dir) if args.save_outputs else None
if out_dir is not None:
    out_dir.mkdir(parents=True, exist_ok=True)

print(args)
print(f"device={device}, dtype={dtype}")
print(f"rae_repo_path={args.rae_repo_path}, rae_dtype={rae_dtype}")
print(f"save_outputs={args.save_outputs}, out_dir={out_dir}")


## 2. 简洁 E / D / V 接口

这一格是推荐入口：`E` 是 encoder，`D` 是 decoder，输入图像叫 `x`，latent 叫 `z`。`R/T/S/N/Ch/Mix` 都是可以直接作用在 `z` 上的短操作子。


In [ ]:
vae = load_vae(args, device, dtype)

VAE_SPECS = {
    "active": {
        "label": "active",
        "source": args.vae_source,
        "hf_model_name": args.hf_model_name,
        "ldm_config": args.ldm_config,
        "ldm_ckpt": args.ldm_ckpt,
        "vae_scaling_factor": args.vae_scaling_factor,
    },
    "eqvae": {
        "label": "eqvae",
        "source": "hf",
        "hf_model_name": "zelaki/eq-vae",
        "vae_scaling_factor": None,
    },
    "eqvae_ema": {
        "label": "eqvae_ema",
        "source": "hf",
        "hf_model_name": "zelaki/eq-vae-ema",
        "vae_scaling_factor": None,
    },
    "sdvae": {
        "label": "sdvae_ft_mse",
        "source": "hf",
        "hf_model_name": "stabilityai/sd-vae-ft-mse",
        "vae_scaling_factor": 0.18215,
    },
    "sdvae_ft_mse": {
        "label": "sdvae_ft_mse",
        "source": "hf",
        "hf_model_name": "stabilityai/sd-vae-ft-mse",
        "vae_scaling_factor": 0.18215,
    },
    "sdvae_ft_ema": {
        "label": "sdvae_ft_ema",
        "source": "hf",
        "hf_model_name": "stabilityai/sd-vae-ft-ema",
        "vae_scaling_factor": 0.18215,
    },
    "sdvae_klf8": {
        "label": "sdvae_klf8",
        "source": "ldm",
        "ldm_config": args.ldm_config,
        "ldm_ckpt": args.sdvae_klf8_ckpt,
        "vae_scaling_factor": 0.18215,
    },
}
VAE_CACHE = {"active": vae}
for key, spec in VAE_SPECS.items():
    if key == "active":
        continue
    same_hf = (
        args.vae_source == "hf"
        and spec.get("source") == "hf"
        and args.hf_model_name == spec.get("hf_model_name")
        and args.vae_scaling_factor == spec.get("vae_scaling_factor", None)
    )
    same_ldm = (
        args.vae_source == "ldm"
        and spec.get("source") == "ldm"
        and args.ldm_config == spec.get("ldm_config")
        and args.ldm_ckpt == spec.get("ldm_ckpt")
        and args.vae_scaling_factor == spec.get("vae_scaling_factor", None)
    )
    if same_hf or same_ldm:
        VAE_CACHE[key] = vae


def get_vae(key="active"):
    """按名字加载并缓存 VAE；可用 key: active, eqvae, eqvae_ema, sdvae, sdvae_ft_ema, sdvae_klf8。"""
    if key in VAE_CACHE:
        return VAE_CACHE[key]
    if key not in VAE_SPECS:
        raise KeyError(f"未知 VAE: {key}; 可选 {list(VAE_SPECS)}")

    spec = VAE_SPECS[key]
    local_args = SimpleNamespace(**vars(args))
    local_args.vae_source = spec["source"]
    local_args.hf_model_name = spec.get("hf_model_name", args.hf_model_name)
    local_args.ldm_config = spec.get("ldm_config", args.ldm_config)
    local_args.ldm_ckpt = spec.get("ldm_ckpt", args.ldm_ckpt)
    local_args.vae_scaling_factor = spec.get("vae_scaling_factor", None)
    VAE_CACHE[key] = load_vae(local_args, device, dtype)
    return VAE_CACHE[key]


def E_with(key, x, posterior=None):
    return get_vae(key).encode(x, posterior=posterior or args.posterior)


def D_with(key, z):
    return get_vae(key).decode(z)


def E_eqvae(x, posterior=None):
    return E_with("eqvae", x, posterior)


def D_eqvae(z):
    return D_with("eqvae", z)


def E_eqvae_ema(x, posterior=None):
    return E_with("eqvae_ema", x, posterior)


def D_eqvae_ema(z):
    return D_with("eqvae_ema", z)


def E_sdvae(x, posterior=None):
    return E_with("sdvae", x, posterior)


def D_sdvae(z):
    return D_with("sdvae", z)


def E_sdvae_ft_mse(x, posterior=None):
    return E_with("sdvae_ft_mse", x, posterior)


def D_sdvae_ft_mse(z):
    return D_with("sdvae_ft_mse", z)


def E_sdvae_ft_ema(x, posterior=None):
    return E_with("sdvae_ft_ema", x, posterior)


def D_sdvae_ft_ema(z):
    return D_with("sdvae_ft_ema", z)


def E_sdvae_klf8(x, posterior=None):
    return E_with("sdvae_klf8", x, posterior)


def D_sdvae_klf8(z):
    return D_with("sdvae_klf8", z)


RAE_CACHE = {}


def get_rae(key):
    """按名字加载并缓存 RAE；可用 key: rae_dinov2, rae_mae, rae_siglip2。"""
    if key in RAE_CACHE:
        return RAE_CACHE[key]
    if key not in RAE_BASELINE_SPECS:
        raise KeyError(f"未知 RAE: {key}; 可选 {list(RAE_BASELINE_SPECS)}")
    RAE_CACHE[key] = load_rae_adapter(
        key,
        repo_path=args.rae_repo_path,
        device=device,
        dtype=rae_dtype,
        auto_clone=args.rae_auto_clone,
        auto_download=args.rae_auto_download,
    )
    return RAE_CACHE[key]


def E_rae_with(key, x):
    return get_rae(key).encode(x)


def D_rae_with(key, z):
    return get_rae(key).decode(z)


def E_rae_dinov2(x):
    return E_rae_with("rae_dinov2", x)


def D_rae_dinov2(z):
    return D_rae_with("rae_dinov2", z)


def E_rae_mae(x):
    return E_rae_with("rae_mae", x)


def D_rae_mae(z):
    return D_rae_with("rae_mae", z)


def E_rae_siglip2(x):
    return E_rae_with("rae_siglip2", x)


def D_rae_siglip2(z):
    return D_rae_with("rae_siglip2", z)


def E_baseline(key, x, posterior=None):
    if key in RAE_BASELINE_SPECS:
        return E_rae_with(key, x)
    return E_with(key, x, posterior)


def D_baseline(key, z):
    if key in RAE_BASELINE_SPECS:
        return D_rae_with(key, z)
    return D_with(key, z)


def check_baselines(keys=None):
    """只检查 baseline 是否配置/下载就绪，不加载大模型。"""
    if keys is None:
        keys = list(VAE_SPECS) + list(RAE_BASELINE_SPECS)
    rae_status = get_rae_status(args.rae_repo_path)
    rows = []
    for key in keys:
        if key in VAE_SPECS:
            spec = VAE_SPECS[key]
            if spec.get("source") == "ldm":
                ckpt = spec.get("ldm_ckpt")
                ready = bool(ckpt) and Path(ckpt).exists()
                note = ckpt if ready else f"缺 checkpoint: {ckpt}"
            else:
                ready = True
                note = spec.get("hf_model_name")
            rows.append((key, "VAE", ready, note))
        elif key in RAE_BASELINE_SPECS:
            s = rae_status[key]
            ready = s["repo"] and s["config"] and s["weights"]
            if ready:
                note = "ready"
            elif not s["repo"]:
                note = f"缺 RAE repo: {args.rae_repo_path}"
            elif not s["config"]:
                note = "缺 config"
            else:
                note = "缺文件: " + "; ".join(s["missing"][:2])
                if len(s["missing"]) > 2:
                    note += f" ...(+{len(s['missing']) - 2})"
            rows.append((key, "RAE", ready, note))
        else:
            rows.append((key, "?", False, "未知 key"))
    for key, kind, ready, note in rows:
        print(f"{key:14s} {kind:3s} {'ready' if ready else 'missing':7s} {note}")
    return rows


def compare_vaes(x, vae_keys=("eqvae", "eqvae_ema"), ops=None, cell_size=192, columns=4):
    """对多个 VAE 做 encode -> latent 操作 -> decode 对比；默认不保存，只返回显示网格。"""
    if ops is None:
        ops = [
            ("D(z)", lambda z: z),
            ("D(R(z,90))", lambda z: R(z, 90)),
            ("D(T(z,4,0))", lambda z: T(z, 4, 0)),
        ]

    items = [("x", x)]
    for key in vae_keys:
        z_key = E_with(key, x)
        items.append((f"{key}:z", z_key))
        for op_name, op in ops:
            items.append((f"{key}:{op_name}", D_with(key, op(z_key))))
    return V(items, cell_size=cell_size, columns=columns)


def compare_baselines(x, keys=("eqvae", "eqvae_ema"), ops=None, cell_size=192, columns=4):
    """统一比较 VAE/RAE tokenizer；key 可混用 eqvae、sdvae、rae_dinov2 等。"""
    if ops is None:
        ops = [
            ("D(z)", lambda z: z),
            ("D(R(z,90))", lambda z: R(z, 90)),
            ("D(T(z,1,0))", lambda z: T(z, 1, 0)),
        ]

    items = [("x", x)]
    for key in keys:
        z_key = E_baseline(key, x)
        items.append((f"{key}:z", z_key))
        for op_name, op in ops:
            items.append((f"{key}:{op_name}", D_baseline(key, op(z_key))))
    return V(items, cell_size=cell_size, columns=columns)


def _indices_to_string(indices):
    if indices is None:
        return None
    if isinstance(indices, str):
        return indices
    if isinstance(indices, int):
        return str(indices)
    return ",".join(str(i) for i in indices)


def pick_x(index=None, indices=None, count=None, seed=None, show=False):
    """从当前输入源取图并返回 x；CIFAR 下支持单图 index、多图 indices/count、随机 seed。"""
    global selected_name, selected_names

    local_args = SimpleNamespace(**vars(args))
    if local_args.input_source == "cifar":
        chosen_indices = _indices_to_string(indices)
        if index is not None:
            chosen_indices = str(index)
        if chosen_indices is not None:
            local_args.indices = chosen_indices
        if count is not None:
            local_args.num_images = count
        if seed is not None and not local_args.indices:
            local_args.seed = seed
    elif indices is not None or count not in (None, 1):
        raise ValueError("文件输入目前只支持单张；多图请把 input_source 设为 'cifar'。")

    images = load_input_images(local_args)
    selected_names = [name for name, _ in images]
    selected_name = selected_names[0]
    x_tensor = torch.cat([image_to_tensor(image, vae.device, vae.dtype) for _, image in images], dim=0)
    if show:
        V(x_tensor, title="x")
    return x_tensor


def E(x, posterior=None):
    """Encoder: x -> z。默认使用 active VAE。"""
    return E_with("active", x, posterior)


def D(z):
    """Decoder: z -> x_hat。默认使用 active VAE。"""
    return D_with("active", z)


def R(z, degrees=90):
    """Rotate: 在 latent 层旋转。90 的倍数会用精确 rot90。"""
    return rotate_latent(z, degrees)


def T(z, dx=0, dy=0):
    """Translate: 在 latent 层零填充平移；dx 向右，dy 向下。"""
    return translate_latent(z, dx, dy)


def Roll(z, dx=0, dy=0):
    """Circular translate: latent 循环平移，会从另一边卷回来。"""
    return torch.roll(z, shifts=(dy, dx), dims=(-2, -1))


def S(z, scale=1.0):
    """Scale: latent 整体缩放。"""
    return z * scale


def N(z, sigma=0.1, seed=None):
    """Noise: 给 latent 加高斯噪声。"""
    if seed is None:
        noise = torch.randn_like(z)
    else:
        generator = torch.Generator(device=z.device).manual_seed(seed)
        noise = torch.randn(z.shape, generator=generator, device=z.device, dtype=z.dtype)
    return z + sigma * noise


def Ch(z, channel=0, amount=1.0):
    """Channel shift: 只移动一个 latent 通道。"""
    out = z.clone()
    out[:, channel] += amount
    return out


def Mix(z1, z2, alpha=0.5):
    """Interpolate: 两个 latent 或两个 latent batch 线性插值。"""
    return z1 * (1 - alpha) + z2 * alpha


def _batch_names(base, count):
    return [base] if count == 1 else [f"{base}_{i}" for i in range(count)]


def _tensor_to_pils_m11_batch(t):
    t = t.detach().float().cpu()
    if t.ndim == 3:
        t = t.unsqueeze(0)
    if t.shape[1] == 1:
        t = t.repeat(1, 3, 1, 1)
    t = t.clamp(-1, 1)
    t = ((t + 1.0) * 127.5).round().byte()
    return [Image.fromarray(t[i].permute(1, 2, 0).numpy(), mode="RGB") for i in range(t.shape[0])]


def _tensor_to_pils_01_batch(t):
    t = t.detach().float().cpu()
    if t.ndim == 3:
        t = t.unsqueeze(0)
    if t.shape[1] == 1:
        t = t.repeat(1, 3, 1, 1)
    t = t.clamp(0, 1)
    t = (t * 255.0).round().byte()
    return [Image.fromarray(t[i].permute(1, 2, 0).numpy(), mode="RGB") for i in range(t.shape[0])]


def _as_images(obj, name="item", mode="auto"):
    if isinstance(obj, Image.Image):
        return [(name, obj.convert("RGB"))]
    if isinstance(obj, torch.Tensor):
        t = obj.detach()
        if t.ndim == 3:
            t = t.unsqueeze(0)
        if t.ndim != 4:
            raise ValueError(f"V 只能显示 PIL 图像或 3/4 维 tensor，当前 shape={tuple(obj.shape)}")

        if mode == "image01":
            images = _tensor_to_pils_01_batch(t)
        elif mode == "first3":
            images = _tensor_to_pils_01_batch(first3_to_rgb(t))
        elif mode == "latent" or t.shape[1] not in (1, 3):
            images = _tensor_to_pils_01_batch(pca_to_rgb(t))
        else:
            images = _tensor_to_pils_m11_batch(t)
        return list(zip(_batch_names(name, len(images)), images))
    raise TypeError(f"V 不支持类型：{type(obj)}")


def V(obj, x=None, title=None, mode="auto", cell_size=256, columns=None):
    """Visualize: V(x)、V(z)、V(D(z)) 都可以；batch 会自动展开成多张图。"""
    items = []
    if x is not None:
        items.extend(_as_images(x, "x"))

    if isinstance(obj, (list, tuple)) and not isinstance(obj, torch.Tensor):
        for i, item in enumerate(obj):
            if isinstance(item, tuple) and len(item) == 2:
                name, value = item
            else:
                name, value = f"item_{i}", item
            items.extend(_as_images(value, str(name), mode=mode))
    else:
        if title is None and isinstance(obj, torch.Tensor):
            title = "z" if obj.ndim >= 4 and obj.shape[1] not in (1, 3) else "x"
        items.extend(_as_images(obj, title or "item", mode=mode))

    cols = columns or min(4, len(items))
    grid = make_grid(items, columns=cols, cell_size=cell_size)
    display(grid)
    return grid



print("E/D/V 与 VAE/RAE baseline 接口已就绪。可先运行 check_baselines() 查看本机权重状态。")


## 3. 选图、处理、画图

这一格是主要实验区：选 `x`，算 `z = E_xxx(x)`，对 `z` 做操作，再用对应的 `D_xxx` 解码和 `V(...)` 画图。复杂封装都在上一格。


In [ ]:
# 1. 选图。保留一行即可，其他示例按需取消注释。
x = pick_x(count=args.num_images, seed=args.seed)       # 从 CIFAR 随机选多张
# x = pick_x(index=123)                                 # 从 CIFAR 指定单张
# x = pick_x(indices=[0, 5, 9])                         # 从 CIFAR 指定多张

# 2. 选择 VAE 并编码。每个 VAE 都有自己的 E_xxx / D_xxx。
z_eqvae = E_eqvae(x)
z_ema = E_eqvae_ema(x)
# z_sdvae = E_sdvae(x)                                  # sdvae 是 stabilityai/sd-vae-ft-mse 的短别名
# z_sdvae_ema = E_sdvae_ft_ema(x)
# z_sdvae_klf8 = E_sdvae_klf8(x)                        # exact LDM baseline，需要本机已有 kl-f8/model.ckpt
# z_rae_dinov2 = E_rae_dinov2(x)                        # 需要 RAE 官方仓库和对应 decoder/stat 权重
# z_rae_mae = E_rae_mae(x)
# z_rae_siglip2 = E_rae_siglip2(x)

# 默认 z 仍然留给你随手操作。
z = z_eqvae

# 3. 对 latent 做操作，再用对应 decoder 解码。
z_rotate = R(z, 90)
z_translate = T(z, 4, 0)
z_noise = N(z, 0.05, seed=seed)

print(f"当前样本: {selected_names}")
print(f"x: {tuple(x.shape)}")
print(f"z_eqvae: {tuple(z_eqvae.shape)}")
print("常用写法：V(z)、V(D_eqvae(z))、V(D_sdvae(E_sdvae(x)))、V(D_rae_dinov2(E_rae_dinov2(x)))")

# 4. 画图。V 会自动展开 batch。
V([
    ("x", x),
    ("eqvae:D(z)", D_eqvae(z_eqvae)),
    ("eqvae:D(R(z,90))", D_eqvae(z_rotate)),
    ("eqvae:D(T(z,4,0))", D_eqvae(z_translate)),
    ("eqvae:D(N(z,.05))", D_eqvae(z_noise)),
    ("eqvae_ema:D(z)", D_eqvae_ema(z_ema)),
    ("eqvae:z", z_eqvae),
    ("eqvae_ema:z", z_ema),
], cell_size=160, columns=4)

# 5. 多 baseline 对比。需要 SD-VAE/RAE 时，把对应 key 加进列表。
baseline_keys_to_show = ["eqvae", "eqvae_ema"]
# baseline_keys_to_show = ["eqvae", "eqvae_ema", "sdvae", "sdvae_ft_ema"]
# baseline_keys_to_show = ["eqvae", "sdvae_klf8", "rae_dinov2"]
# baseline_keys_to_show = ["eqvae", "sdvae", "rae_dinov2", "rae_mae", "rae_siglip2"]
compare_baselines(x, baseline_keys_to_show, cell_size=160, columns=4)
